# Smart Buildings — Run 4 Ablation (Deadbanded Soft Guard + 6-Climate Diverse Synthetic Suite)

5 shipped scenarios (8640 steps x 3 arms) + 6 diverse unseen scenarios across all 6 climates (8640 steps x 2 arms) with 4 parallel workers.
Results land in `/kaggle/working/ablation_results/`.

In [ ]:
import pathlib, subprocess, os

URL  = "https://github.com/KAC-PDW/smart-buildings-competition.git"
REPO = pathlib.Path("/kaggle/working/smart-buildings-competition")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", URL, str(REPO)], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
!pip install -q -e . --no-deps

try:
    import numpy, numba
    print("numpy", numpy.__version__, "| numba ok")
except Exception as exc:
    print("numba broken, pinning numpy 2.0.2:", exc)
    !pip install -q "numpy==2.0.2"

import os
print("cpu count:", os.cpu_count())

In [ ]:
# 288-step smoke check (arms A/B with 2 workers)
!python smart_buildings_toolkit/scripts/run_experiment.py --mode smoke --arms A B --workers 2 --skip-install

In [ ]:
# Full Run 4 ablation across shipped + unseen_diverse (6 climates) splits
!python smart_buildings_toolkit/scripts/run_experiment.py --mode run4 --extra-test-count 6 --workers 4 --skip-install

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/working/ablation_results/ablation_summary.csv")
for split_name, sub in df.groupby("split"):
    print(f"=== {split_name} ===")
    piv = sub.pivot_table(index=["building", "weather"], columns="arm", values="mae_vs_zero")
    print(piv.round(5).to_string())
    print()
    print(sub.groupby("arm")["mae_vs_zero"].mean().round(5).to_string())
    print()